In [0]:
# CELL: Install BeautifulSoup, Python-Dotenv
#-----------------------------------------------------------

# 1. Install BeautifulSoup
%pip install beautifulsoup4

# 2. Verify Installation
try:
    from bs4 import BeautifulSoup
    print("BeautifulSoup is installed and working correctly.")
except ImportError:
    raise ImportError(
        "BeautifulSoup failed to install.  ",  
        "Please rerun this cell"
    )

# 3. Install Python-Dotenv
%pip install python-dotenv

# 4. Verify Installation
try:
    from dotenv import load_dotenv
    print("Dotenv is installed and working correctly.")
except ImportError:
    raise ImportError(
        "Dotenv failed to install.  ", 
        "Please rerun this cell",
    )

In [0]:
# CELL: Import Required Libraries
#-----------------------------------------------------------

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import requests

In [0]:
# CELL: Load Environment Variables
#-----------------------------------------------------------

# 1. Load environment variables from file
load_dotenv()

# 2. Assign storage acount name and key vars from env
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

# 3. Verify env read successfully
print(f"storage_account_name = {storage_account_name}")

In [0]:
# CELL: Configure spark settings
#-----------------------------------------------------------

# 1. AUTO-CREATE-FIX: Configuration auto-creates the 'bronze' container if it is missing
spark.conf.set(
    f"fs.azure.createRemoteFileSystemDuringInitialization", "true"
)

# 2. Connect to storage account
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

print("Configuration loaded & Auto-create enabled")

In [0]:
# CELL: Assign constants
#-----------------------------------------------------------

# 1. Assign zones 
ZN_TRANSACTIONS = "transactions"
ZN_UNITS = "units"
ZN_PROJECTS = "projects"

# 2. Assign base path for zones
ZONES_BASE_PATH = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

# 3. Assign zone destinations
DEST_LANDING = "landing"
DEST_ARCHIVE = "archive"

# 4. Assign csv file name
UNITS_CSV = f"{ZN_UNITS}.csv"
PROJECTS_CSV = f"{ZN_PROJECTS}.csv"
CURRENT_CSV = "current.csv"

# 5. Assign requests headers
REQUESTS_HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# 6. Assign base_page_url
BASE_PAGE_URL = "https://www.dubaipulse.gov.ae/data/dld-registration/"

# 7. Assign zone page URLs
ZN_PAGE_URLS = {zone: f"{BASE_PAGE_URL}dld_{zone}-open" for zone in [ZN_UNITS, ZN_PROJECTS]}

# 8. Assign chunk size
TEN_MBYTES = 1024 * 1024 * 10


In [0]:
# CELL: Define Functions
#-----------------------------------------------------------

def ingest_dataset(dataset_name: str, page_url: str, file_pattern: str):
    """
    Ingests file for the provided dataset_name from the provided
    page_url to the appropriate landing path for the provided
    dataset
    Args:
        dataset_name (str): The dataset to be ingested
        page_url (str): The URL from which the dataset will be ingested
        file_pattern (str): The file_pattern to search for from the page_url source
    Returns:
        None    
    """
    print(f"\nSTARTING INGESTION: {dataset_name.upper()}")

    # Define dynamic paths (bronze/unit|project|...)
    base_path = f"{ZONES_BASE_PATH}{dataset_name}/"
    landing_zone = f"{base_path}{DEST_LANDING}/"
    archive_zone = f"{base_path}{DEST_ARCHIVE}/"

    # A. Link Finder
    print(f"Finding download link on page '{page_url}'...")
    rsp = requests.get(page_url, headers=REQUESTS_HEADERS)
    soup = BeautifulSoup(rsp.content, "html.parser")

    fresh_url = None
    for link in soup.find_all("a", href=True):
        link_href = link["href"]
        if "download" in link_href and file_pattern in link_href:
            fresh_url = f"{BASE_PAGE_URL}{link_href}"
            break

    if not fresh_url:
        raise Exception(f"Unable to find download link for {file_pattern} on {page_url}")

    # B. Archive Current File (if exists)
    try:
        dbutils.fs.ls(f"{landing_zone}{CURRENT_CSV}")
        today = datetime.now().strftime("%Y-%m-%d")
        dbutils.fs.mv(f"{landing_zone}{CURRENT_CSV}", f"{archive_zone}{today}.csv")
        print(f"Archived previous current file to {archive_zone}{today}.csv")
    
    except: pass

    # C. Download & Upload
    local_path = f"/tmp/{dataset_name}.csv"
    print("Downloading & uploading...")
    
    with requests.get(fresh_url, headers=REQUESTS_HEADERS, stream=True) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=TEN_MBYTES):
                f.write(chunk)
    
    dbutils.fs.mv(f"file:{local_path}", f"{landing_zone}{CURRENT_CSV}")
    os.remove(local_path)

    print(f"FINISHED: {dataset_name} is ready.")


print("Master Function Loaded")


In [0]:
# CELL: Execute Ingestion
#-----------------------------------------------------------

# 1. Ingest Units (Product Dimension)
ingest_dataset(dataset_name=ZN_UNITS, page_url=ZN_PAGE_URLS[ZN_UNITS], file_pattern=UNITS_CSV)

# 2. Ingest Projects (Status Dimension)
ingest_dataset(dataset_name=ZN_PROJECTS, page_url=ZN_PAGE_URLS[ZN_PROJECTS], file_pattern=PROJECTS_CSV)

In [0]:
# CELL: Verify Dimension Files
#-----------------------------------------------------------

print("Verifying dimension files in Bronze layer...")

files_to_check = {zone:f"{ZONES_BASE_PATH}{zone}/{DEST_LANDING}/{CURRENT_CSV}" for zone in [ZN_UNITS, ZN_PROJECTS]}

all_good = True
for dataset, file_path in files_to_check.items():
    try:
        dbutils.fs.ls(file_path)
        print(f"File exists: {dataset}/{DEST_LANDING}/{CURRENT_CSV}")
    except Exception as e:
        print(f"File does not exist: {dataset}/{DEST_LANDING}/{CURRENT_CSV}")
        all_good = False

assert all_good, "One or more files are missing from the Bronze layer. Please upload them manually"

print("All dimension files verified! Pipeline can proceed.")